In [1]:
# Import thư viện
import os
import shutil
from pathlib import Path
from collections import Counter
from PIL import Image, ImageOps

In [2]:
# Input và Output
DATA_DIR   = "Dataset 2/wastes"
OUTPUT_DIR = "Dataset 2_clean"

if Path(OUTPUT_DIR).exists(): 
    shutil.rmtree(OUTPUT_DIR)

MIN_SIZE    = 64          # Ngưỡng tối thiểu (px), ảnh nhỏ hơn sẽ bị xóa
TARGET_SIZE = (256, 256)  # Kích thước chuẩn, đồng nhất với Dataset 1
IMAGE_EXTS  = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

In [3]:
# Convert RGBA -> RGB
def rgba_to_rgb(img: Image.Image) -> Image.Image:
    bg = Image.new("RGB", img.size, (255, 255, 255))
    bg.paste(img, mask=img.split()[3])   
    return bg

In [4]:
# Hàm xử lý một ảnh
def process_image(src_path: Path, dst_path: Path) -> str:
    try:
        img = Image.open(src_path)

        # Loại bỏ ảnh grayscale (mode L)
        if img.mode == "L": return "dropped_L"
 
        # Loại bỏ ảnh quá nhỏ (outlier)
        if img.size[0] < MIN_SIZE or img.size[1] < MIN_SIZE:
            return "dropped_small"
        
        # Chuẩn hóa mode màu về RGB
        if img.mode == "RGBA": 
            img = rgba_to_rgb(img)
        elif img.mode != "RGB": 
            img = img.convert("RGB")
 
        # Resize về 256x256 
        img = ImageOps.fit(img, TARGET_SIZE, method=Image.LANCZOS)
 
        # Lưu thành PNG
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        img.save(dst_path, "PNG")
 
        return "kept"
 
    except Exception as e:
        return f"error: {e}"

In [5]:
def clean_dataset(data_dir: str, output_dir: str):
    data_dir   = Path(data_dir)
    output_dir = Path(output_dir)
 
    splits = ["train", "test"]
    # Đếm toàn bộ dataset
    stats_total = Counter()
 
    for split in splits:
        split_dir = data_dir / split
 
        if not split_dir.exists():
            print(f" Không tìm thấy thư mục: {split_dir}")
            continue
 
        classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
        print(f"\n[{split.upper()}] {len(classes)} class")
        print(f"  {'Class':<30} {'Tổng':>6}  {'Giữ':>6}  {'Bỏ_L':>6}  {'Bỏ_nhỏ':>8}  {'Lỗi':>5}")
 
        stats_split = Counter()
 
        for cls in classes:
            cls_src_dir = split_dir / cls
            cls_dst_dir = output_dir / split / cls
 
            # Lấy danh sách ảnh hợp lệ
            img_files = [
                f for f in cls_src_dir.iterdir()
                if f.is_file() and f.suffix.lower() in IMAGE_EXTS
            ]
 
            cls_stats = Counter()
 
            for src_path in img_files:
                # Tên file đầu ra luôn là .png
                dst_name = src_path.stem + ".png"
                dst_path = cls_dst_dir / dst_name
 
                result = process_image(src_path, dst_path)
                cls_stats[result] += 1
 
            stats_split += cls_stats
            stats_total += cls_stats
 
            kept        = cls_stats["kept"]
            dropped_L   = cls_stats["dropped_L"]
            dropped_s   = cls_stats["dropped_small"]
            errors      = sum(v for k, v in cls_stats.items() if k.startswith("error"))
            total_cls   = len(img_files)
 
            print(f"  {cls:<30} {total_cls:>6}  {kept:>6}  {dropped_L:>6}  {dropped_s:>8}  {errors:>5}")
 
        kept_s      = stats_split["kept"]
        dropped_L_s = stats_split["dropped_L"]
        dropped_s_s = stats_split["dropped_small"]
        errors_s    = sum(v for k, v in stats_split.items() if k.startswith("error"))
        print(f"  {'TỔNG ' + split:<30} {sum(stats_split.values()):>6}  {kept_s:>6}  {dropped_L_s:>6}  {dropped_s_s:>8}  {errors_s:>5}") 
    # Tổng kết
    kept_t      = stats_total["kept"]
    dropped_L_t = stats_total["dropped_L"]
    dropped_s_t = stats_total["dropped_small"]
    errors_t    = sum(v for k, v in stats_total.items() if k.startswith("error"))
 
    print(f"  KẾT QUẢ TOÀN BỘ")
    print(f"  Ảnh giữ lại          : {kept_t}")
    print(f"  Bỏ do mode L         : {dropped_L_t}")
    print(f"  Bỏ do ảnh quá nhỏ   : {dropped_s_t}  (< {MIN_SIZE}px)")
    print(f"  Lỗi                  : {errors_t}")
    print(f"  Đã lưu vào           : {output_dir.resolve()}")

In [6]:
if __name__ == "__main__":
    clean_dataset(DATA_DIR, OUTPUT_DIR)


[TRAIN] 9 class
  Class                            Tổng     Giữ    Bỏ_L    Bỏ_nhỏ    Lỗi
  E-waste                          1248    1246       0         2      0
  automobile wastes                 871     868       3         0      0
  battery waste                     848     848       0         0      0
  glass waste                      1022    1022       0         0      0
  light bulbs                       420     416       4         0      0
  metal waste                      1231    1229       2         0      0
  organic waste                     889     889       0         0      0
  paper waste                      1370    1359      11         0      0
  plastic waste                    1315    1315       0         0      0
  TỔNG train                       9214    9192      20         2      0

[TEST] 9 class
  Class                            Tổng     Giữ    Bỏ_L    Bỏ_nhỏ    Lỗi
  E-waste                           313     313       0         0      0
  automobile waste